# OSA 온톨로지 구축 — 1·2단계: 변수 인벤토리 파이프라인

SHHS(v0.14.0)·MESA(v0.8.0) 변수 사전에서 수면무호흡(OSA) 관련 변수를 선별·분류하고,
고조합 PSG 변수를 패싯(facet) 구조로 축약하는 파이프라인.

**사용법**: Kernel → Restart & Run All. 데이터 경로는 [2] 셀에서 수정.

**산출물** (`ontology_outputs/`):
| 파일 | 내용 |
|---|---|
| `OSA_variable_inventory_v2.xlsx` | 변수 분류(포함/축약/제외) 인벤토리 |
| `OSA_facet_decomposition.xlsx` | 축약 변수의 6차원 패싯 분해 |
| `MESA_SHHS_id_correspondence.xlsx` | 코호트 간 id 수준 직접 대응 |
| `OSA_ontology_class_design.xlsx` | 측정 클래스(패밀리) + 패싯 속성 설계표 |


## [1] 환경 확인

In [1]:
import os, re, sys
from pathlib import Path
import pandas as pd
print(sys.version)
print("pandas", pd.__version__)

3.11.15 (main, Mar 11 2026, 17:20:07) [GCC 14.3.0]
pandas 3.0.3


## [2] 데이터 경로 및 파일 탐색
대용량 폴더(PSG 신호, archive 중복본)는 진입하지 않음.

In [2]:
SHHS_DIR = Path("/mnt/y/Dataset/shhs")   # 필요시 수정
MESA_DIR = Path("/mnt/y/Dataset/mesa")   # 필요시 수정
OUT_DIR = Path("ontology_outputs"); OUT_DIR.mkdir(exist_ok=True)

SKIP_DIRS = {"archive", "polysomnography", "edfs", "annotations-events-nsrr",
             "annotations-events-profusion", "actigraphy", "overlap"}

def scan_csv(base: Path):
    found = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d.lower() not in SKIP_DIRS]
        found += [Path(root) / f for f in files
                  if f.lower().endswith((".csv", ".xlsx", ".txt"))]
    return found

all_files = {n: scan_csv(b) for n, b in [("SHHS", SHHS_DIR), ("MESA", MESA_DIR)]}
{n: len(v) for n, v in all_files.items()}

{'SHHS': 17, 'MESA': 5}

## [3] 최신 버전 변수 사전 로드

In [3]:
def pick_latest(files, must_contain):
    cands = [p for p in files if all(k in p.name.lower() for k in must_contain)]
    def ver(p):
        m = re.search(r"(\d+)\.(\d+)\.(\d+)", p.name)
        return tuple(map(int, m.groups())) if m else (0, 0, 0)
    return max(cands, key=ver) if cands else None

dd = {}
for name in all_files:
    dd[name] = {}
    for kind in ("variables", "domains", "forms"):
        p = pick_latest(all_files[name], ["data-dictionary", kind])
        if p is not None:
            dd[name][kind] = pd.read_csv(p, low_memory=False)
            print(f"{name} {kind}: {p.name}  ({dd[name][kind].shape[0]:,}행)")

SHHS variables: shhs-data-dictionary-0.14.0-variables.csv  (1,993행)
SHHS domains: shhs-data-dictionary-0.14.0-domains.csv  (318행)
SHHS forms: shhs-data-dictionary-0.14.0-forms.csv  (21행)
MESA variables: mesa-data-dictionary-0.8.0-variables.csv  (665행)
MESA domains: mesa-data-dictionary-0.8.0-domains.csv  (142행)
MESA forms: mesa-data-dictionary-0.8.0-forms.csv  (2행)


## [4] 폴더 기반 분류 정책 → 인벤토리 v2
- **개별**: 변수별 온톨로지 클래스 후보
- **축약**: 패싯 조합 변수 → 패밀리로 축약 모델링
- **부분**: 폴더 내 키워드 선별(약물·Interim·MESA 인구학)
- **제외**: v1 범위 밖 (Spectral/HRV/Actigraphy/SF-36/Signal Quality 등)

주의: SHHS display_name에는 스터디 이름("Sleep Heart Health Study")이 붙어 있어
선별 키워드 매칭 전에 반드시 제거해야 함(`STRIP_STUDY`).

In [4]:
SHHS_POLICY = [
    ("Measurements/Polysomnography/Respiratory Events/Apnea/Central",     "축약"),
    ("Measurements/Polysomnography/Respiratory Events/Apnea/Obstructive", "축약"),
    ("Measurements/Polysomnography/Respiratory Events/Hypopnea",          "축약"),
    ("Measurements/Polysomnography/Respiratory Events/Indexes",           "축약"),
    ("Measurements/Polysomnography/Oxygen Saturation",                    "축약"),
    ("Measurements/Polysomnography/Sleep Architecture",                   "개별"),
    ("Measurements/Polysomnography/Arousals",                             "개별"),
    ("Measurements/Polysomnography/Heart Rate",                           "제외"),
    ("Measurements/Polysomnography/Signal Quality",                       "제외"),
    ("Measurements/Polysomnography/Medical Alert",                        "제외"),
    ("Measurements/Polysomnography/Administrative",                       "제외"),
    ("Measurements/ECG",                                                  "제외"),
    ("Measurements/Blood Pressure",                                       "개별"),
    ("Measurements/Anthropometry",                                        "개별"),
    ("Measurements/Bloods",                                               "개별"),
    ("Measurements/Lung Function",                                        "개별"),
    ("Measurements/Administrative",                                       "제외"),
    ("Questionnaires/SHHS1/Sleep Habits",                                 "개별"),
    ("Questionnaires/SHHS2/Sleep Habits",                                 "개별"),
    ("Questionnaires/SHHS1/Epworth",                                      "개별"),
    ("Questionnaires/SHHS2/Epworth",                                      "개별"),
    ("Questionnaires/SHHS1/Morning Survey",                               "개별"),
    ("Questionnaires/SHHS2/Morning Survey",                               "개별"),
    ("Questionnaires/SHHS1/Health Interview",                             "개별"),
    ("Questionnaires/SHHS2/Health Interview",                             "개별"),
    ("Questionnaires/SHHS2/FOSQ",                                         "개별"),
    ("Questionnaires/SHHS2/SAQLI",                                        "개별"),
    ("Questionnaires/SHHS1/SF-36",                                        "제외"),
    ("Questionnaires/SHHS2/SF-36",                                        "제외"),
    ("Questionnaires/SHHS1/Quality Of Life",                              "제외"),
    ("Questionnaires/SHHS2/Quality Of Life",                              "제외"),
    ("Questionnaires/SHHS1/Adverse Events",                               "제외"),
    ("Questionnaires/SHHS1/PSG Signal Verification",                      "제외"),
    ("Questionnaires/SHHS2/Physical Measurements",                        "개별"),
    ("Medical History",                                                   "개별"),
    ("CVD Outcomes",                                                      "개별"),
    ("Medications",                                                       "부분"),
    ("Interim",                                                           "부분"),
    ("Demographics",                                                      "개별"),
    ("Spectral Analysis",                                                 "제외"),
    ("HRV Analysis",                                                      "제외"),
    ("Administrative",                                                    "제외"),
]

MESA_POLICY = [
    ("Sleep and Circadian Studies/Polysomnography/Sleep Disordered Breathing/Apnea-Hypopnea Frequency Indices", "축약"),
    ("Sleep and Circadian Studies/Polysomnography/Sleep Disordered Breathing/Oxygen Saturation Metrics",        "축약"),
    ("Sleep and Circadian Studies/Polysomnography/Sleep Disordered Breathing/Respiratory Event Length",         "축약"),
    ("Sleep and Circadian Studies/Polysomnography/Sleep Disordered Breathing/Respiratory Event Counts",         "축약"),
    ("Sleep and Circadian Studies/Polysomnography/Sleep Disordered Breathing/Endotypes",                        "개별"),
    ("Sleep and Circadian Studies/Polysomnography/Sleep Disordered Breathing",                                  "개별"),
    ("Sleep and Circadian Studies/Polysomnography/Sleep Electroencephalogram (EEG)",                            "개별"),
    ("Sleep and Circadian Studies/Polysomnography/Limb Movements",                                              "제외"),
    ("Sleep and Circadian Studies/Polysomnography/Cardiac Data Within Sleep",                                   "제외"),
    ("Sleep and Circadian Studies/Polysomnography/Sigal Quality",                                               "제외"),
    ("Sleep and Circadian Studies/Polysomnography/Administrative",                                              "제외"),
    ("Sleep and Circadian Studies/Actigraphy",                                                                  "제외"),
    ("Physiologic Measurements/Cardiovascular Assessments/Heart Rate Variability",                              "제외"),
    ("Sleep Questionnaires",                                                                                    "개별"),
    ("Sleep Treatment",                                                                                         "개별"),
    ("Harmonized",                                                                                              "개별"),
    ("Anthropometry",                                                                                           "개별"),
    ("Sociodemographics",                                                                                       "개별"),
    ("Lifestyle and Behavioral Health",                                                                         "개별"),
    ("Administrative",                                                                                          "부분"),
]

STRIP_STUDY = re.compile(r"sleep heart health study|multi-?ethnic study of atherosclerosis", re.I)

PARTIAL_KEEP = {
    ("SHHS", "Medications"): re.compile(
        r"angiotensin|\bace\b|\barb\b|beta.?block|alpha.?block|calcium|diuretic|"
        r"statin|lipid|choleste|insulin|diabet|glucosidase|glycemic|sulfonylurea|metformin|"
        r"digoxin|arrhythm|nitrate|warfarin|anticoag|platelet|aspirin|hypert|"
        r"benzo|sedat|hypnot|barbit|\bsleep\b|snor|apnea|cpap|"
        r"amlodipine|\bccbs?\b|biguanide|digitalis|bile.?acid|fibrate|niacin", re.I),
    ("SHHS", "Interim"): re.compile(r"\bsleep\b|snor|apnea|cpap|breath|doze|nap|tired", re.I),
    ("MESA", "Administrative"): re.compile(r"gender|race|age|ethnic", re.I),
}

def classify(name, v):
    policy = SHHS_POLICY if name == "SHHS" else MESA_POLICY
    out = v.copy()
    decisions = []
    for _, row in out.iterrows():
        folder = row["folder"] if pd.notna(row["folder"]) else ""
        dec, matched_prefix = "미분류", None
        for prefix, d in policy:
            if folder.startswith(prefix):
                dec, matched_prefix = d, prefix
                break
        if dec == "부분":
            pat = PARTIAL_KEEP.get((name, matched_prefix))
            text = STRIP_STUDY.sub("", f'{row["id"]} {row["display_name"]} {row["description"]}')
            dec = "개별(선별)" if (pat and pat.search(text)) else "제외(선별탈락)"
        elif dec == "축약" and row.get("commonly_used") is True:
            dec = "개별(주요)"
        decisions.append(dec)
    out["분류"] = decisions
    out["NSRR조화"] = out["folder"].fillna("").str.startswith("Harmonized")
    return out

inv = {}
out_inventory = OUT_DIR / "OSA_variable_inventory_v2.xlsx"
with pd.ExcelWriter(out_inventory, engine="openpyxl") as xw:
    for name in dd:
        c = classify(name, dd[name]["variables"])
        inv[name] = c
        print(f"===== {name} =====")
        print(c["분류"].value_counts().to_string(), "\n")
        un = c[c["분류"] == "미분류"]["folder"].unique()
        if len(un):
            print("!! 미분류 폴더:", list(un))
        for grp, sheet in [(["개별", "개별(주요)", "개별(선별)"], f"{name}_포함"),
                           (["축약"], f"{name}_축약"),
                           (["제외", "제외(선별탈락)", "미분류"], f"{name}_제외")]:
            c[c["분류"].isin(grp)].to_excel(xw, sheet_name=sheet, index=False)
print("저장:", out_inventory.resolve())

===== SHHS =====
분류
축약          764
제외          556
개별          413
제외(선별탈락)    139
개별(선별)       98
개별(주요)       23 

===== MESA =====
분류
제외          372
축약          135
개별          132
개별(주요)       16
제외(선별탈락)      7
개별(선별)        3 

저장: /mnt/e/Claude Cowork/SHHS Ontology/ontology_outputs/OSA_variable_inventory_v2.xlsx


## [5] 축약 변수 패싯 분해
6차원: 측정치 × 이벤트 × 수면단계 × 체위 × desat기준 × 각성기준.

display_name 파싱 주의사항(검증 과정에서 확인된 함정):
- RDI는 설명문에 "oxygen desaturation"이 있어도 호흡장애 종합지수로 분류
- "Average length of X"(MESA)와 "Average X length"(SHHS) 모두 평균길이
- "Total number of" = 개수(빈도), "Maximun"은 MESA 사전의 원본 오타

In [ ]:
def parse_facets(dn):
    t = dn if isinstance(dn, str) else ""
    d = {}
    d["수면단계"] = "NREM" if re.search(r"\bNREM\b", t) else ("REM" if re.search(r"\bREM\b", t) else "전체")
    if re.search(r"non-?supine", t, re.I): d["체위"] = "비앙와위"
    elif re.search(r"\bsupine\b", t, re.I): d["체위"] = "앙와위"
    else: d["체위"] = "전체"
    m = re.search(r">=\s*(\d)\s*%", t)
    m2 = re.search(r"<\s*(\d+)\s*%", t)
    if m: d["desat기준"] = f">={m.group(1)}%"
    elif m2: d["desat기준"] = f"<{m2.group(1)}%"
    elif re.search(r"all oxygen desat|no oxygen desaturation threshold", t, re.I): d["desat기준"] = "전체/무기준"
    else: d["desat기준"] = "미지정"
    d["각성기준"] = "각성포함" if re.search(r"w/ arousals?|or (with )?arousal", t, re.I) else "각성무관"

    EVENTS = [
        (r"Central/Obstructive Apnea ratio", "중추/폐쇄비"),
        (r"Respiratory Disturbance Index|\bRDI\b", "호흡장애종합"),
        (r"Central Apnea-Hypopnea", "중추성무호흡+저호흡"),
        (r"Obstructive Apnea-Hypopnea", "폐쇄성무호흡+저호흡"),
        (r"Apnea-Hypopnea", "무호흡+저호흡"),
        (r"Obstructive Apneas?", "폐쇄성무호흡"),
        (r"Central Apneas?", "중추성무호흡"),
        (r"Hypopneas?", "저호흡"),
        (r"\bapneas?\b", "무호흡(전체)"),
        (r"oxygen desaturations?", "산소탈포화"),
        (r"oxygen saturation|SaO2", "산소포화도"),
    ]
    for pat, ev in EVENTS:
        if re.search(pat, t, re.I):
            d["이벤트"] = ev; break
    else:
        d["이벤트"] = "?"

    MEASURES = [
        (r"\bratio\b", "비율"),
        (r"^Average (length|duration)|^Average .* length", "평균길이"),
        (r"^Longest|^Maxim", "최대"),
        (r"^Minimum|^Min\b|^Shortest", "최소"),
        (r"Index|per hour|/ ?hour", "지수"),   # 시간당 비율(per hour)은 개수가 아니라 지수
        (r"^Total number|^Number", "빈도"),
        (r"^Percent", "시간비율"),
        (r"^Total", "총량"),
        (r"^Average", "평균수준"),
    ]
    for pat, mt in MEASURES:
        if re.search(pat, t, re.I):
            d["측정치"] = mt; break
    else:
        d["측정치"] = "?"
    return d

fac = {}
out_facet = OUT_DIR / "OSA_facet_decomposition.xlsx"
with pd.ExcelWriter(out_facet, engine="openpyxl") as xw:
    for name in inv:
        sub = inv[name][inv[name]["분류"] == "축약"].copy().reset_index(drop=True)
        F = sub["display_name"].map(parse_facets).apply(pd.Series)
        sub = pd.concat([sub, F], axis=1)
        fac[name] = sub
        unk = sub[(sub["이벤트"] == "?") | (sub["측정치"] == "?")]
        print(f"===== {name}: 축약 {len(sub)}개, 파싱 실패 {len(unk)}개 =====")
        if len(unk):
            print(unk[["id", "display_name"]].to_string(index=False))
        sub.to_excel(xw, sheet_name=f"{name}_패싯", index=False)
        (sub.groupby(["측정치", "이벤트", "수면단계", "체위", "desat기준", "각성기준"])
            .size().rename("변수수").reset_index()
            .to_excel(xw, sheet_name=f"{name}_조합", index=False))
print("저장:", out_facet.resolve())

## [6] SHHS ↔ MESA id 수준 직접 대응
NSRR 표준 명명 공유(동일 id) + MESA의 Exam 5 접미사 `5` 규칙.

In [6]:
shhs_ids = set(dd["SHHS"]["variables"]["id"].str.lower())
m = dd["MESA"]["variables"].copy()
m["id_lc"] = m["id"].str.lower()
m["직접일치"] = m["id_lc"].isin(shhs_ids)
m["접미5제거일치"] = ~m["직접일치"] & m["id_lc"].str.replace(r"5$", "", regex=True).isin(shhs_ids)
n1, n2 = int(m["직접일치"].sum()), int(m["접미5제거일치"].sum())
print(f"MESA {len(m)}개 중 SHHS id 대응: {n1 + n2}개 (동일 id {n1} + 접미사'5' 규칙 {n2})")
print(m[m["직접일치"] | m["접미5제거일치"]]["folder"].value_counts().to_string())
m.to_excel(OUT_DIR / "MESA_SHHS_id_correspondence.xlsx", index=False)

MESA 665개 중 SHHS id 대응: 140개 (동일 id 43 + 접미사'5' 규칙 97)
folder
Sleep and Circadian Studies/Polysomnography/Sleep Disordered Breathing/Apnea-Hypopnea Frequency Indices    40
Sleep and Circadian Studies/Polysomnography/Sleep Disordered Breathing/Oxygen Saturation Metrics           32
Sleep and Circadian Studies/Polysomnography/Sigal Quality                                                  30
Sleep and Circadian Studies/Polysomnography/Sleep Electroencephalogram (EEG)/Sleep Architecture            19
Sleep and Circadian Studies/Polysomnography/Sleep Electroencephalogram (EEG)/Arousals                      11
Sleep and Circadian Studies/Polysomnography/Sleep Disordered Breathing/Respiratory Event Length             4
Sleep and Circadian Studies/Polysomnography/Sleep Disordered Breathing/Respiratory Event Counts             3
Administrative                                                                                              1


## [7] 온톨로지 클래스 설계표
패싯 패밀리(측정치×이벤트)별 영문 클래스명 + 패싯 속성 정의 → 3단계(OWL 형식화)의 입력물.

In [7]:
EV_EN = {"폐쇄성무호흡": "ObstructiveApnea", "중추성무호흡": "CentralApnea", "저호흡": "Hypopnea",
         "무호흡+저호흡": "ApneaHypopnea", "중추성무호흡+저호흡": "CentralApneaHypopnea",
         "폐쇄성무호흡+저호흡": "ObstructiveApneaHypopnea", "산소탈포화": "OxygenDesaturation",
         "산소포화도": "OxygenSaturation", "중추/폐쇄비": "CentralToObstructiveApnea",
         "호흡장애종합": "RespiratoryDisturbance", "무호흡(전체)": "Apnea"}
SAT_EVENTS = {"산소포화도", "산소탈포화"}
MT_EN_EVENT = {"지수": "Index", "빈도": "Count", "최대": "MaximumLength", "최소": "MinimumLength",
               "평균길이": "AverageLength", "평균수준": "AverageLevel", "시간비율": "PercentTime",
               "총량": "Total", "비율": "Ratio"}
MT_EN_SAT = {**MT_EN_EVENT, "최대": "Maximum", "최소": "Minimum"}

def class_name(ev, mt):
    mt_en = (MT_EN_SAT if ev in SAT_EVENTS else MT_EN_EVENT).get(mt, "X")
    return f"{EV_EN.get(ev, 'X')}{mt_en}"

fam_rows = []
for (ev, mt), _ in pd.concat([fac["SHHS"], fac["MESA"]]).groupby(["이벤트", "측정치"]):
    sh = fac["SHHS"].query("이벤트 == @ev and 측정치 == @mt")
    me = fac["MESA"].query("이벤트 == @ev and 측정치 == @mt")
    fam_rows.append({"클래스명": class_name(ev, mt), "이벤트": ev, "측정치": mt,
                     "n_SHHS": len(sh), "n_MESA": len(me),
                     "SHHS예시": sh["id"].iloc[0] if len(sh) else "",
                     "MESA예시": me["id"].iloc[0] if len(me) else ""})
fam = pd.DataFrame(fam_rows).sort_values(["n_SHHS", "n_MESA"], ascending=False)

FACET_PROPS = pd.DataFrame([
    {"속성명": "hasSleepStageScope",       "차원": "수면단계",  "허용값": "REMSleep | NREMSleep | AllSleep"},
    {"속성명": "hasBodyPositionScope",     "차원": "체위",     "허용값": "SupinePosition | NonSupinePosition | AllPositions"},
    {"속성명": "hasDesaturationThreshold", "차원": "desat기준", "허용값": "2% | 3% | 4% | 5% | <90%abs | NoThreshold"},
    {"속성명": "hasArousalCriterion",      "차원": "각성기준",  "허용값": "WithArousal | ArousalNotRequired"},
    {"속성명": "hasCohortProvenance",      "차원": "출처",     "허용값": "SHHS1 | SHHS2 | MESA-Exam5"},
])

out_design = OUT_DIR / "OSA_ontology_class_design.xlsx"
with pd.ExcelWriter(out_design, engine="openpyxl") as xw:
    fam.to_excel(xw, sheet_name="측정클래스(패싯패밀리)", index=False)
    FACET_PROPS.to_excel(xw, sheet_name="패싯속성정의", index=False)
    fac["SHHS"].to_excel(xw, sheet_name="SHHS_패싯전체", index=False)
    fac["MESA"].to_excel(xw, sheet_name="MESA_패싯전체", index=False)

n_vars = len(fac["SHHS"]) + len(fac["MESA"])
print(f"측정 클래스(패밀리) {len(fam)}개가 축약 변수 {n_vars}개를 커버")
print(fam.to_string(index=False))
print("저장:", out_design.resolve())

측정 클래스(패밀리) 32개가 축약 변수 899개를 커버
                          클래스명     이벤트  측정치  n_SHHS  n_MESA         SHHS예시      MESA예시
         ObstructiveApneaCount  폐쇄성무호흡   빈도      80       1          oanba      oarop5
                 HypopneaCount     저호흡   빈도      80       0          hnrba            
             CentralApneaCount  중추성무호흡   빈도      80       0          canba            
     OxygenDesaturationMaximum   산소탈포화   최대      40       4         mxdnba     mxdnbp5
OxygenDesaturationAverageLevel   산소탈포화 평균수준      40       4         avdnba     avdnbp5
         HypopneaMaximumLength     저호흡   최대      40       2         mxhnba    longhyp5
         HypopneaAverageLength     저호흡 평균길이      40       2         avhnba    havgdur5
     CentralApneaAverageLength  중추성무호흡 평균길이      40       1        avcanba    cavgdur5
 ObstructiveApneaAverageLength  폐쇄성무호흡 평균길이      40       1        avoanba    oavgdur5
     OxygenDesaturationMinimum   산소탈포화   최소      40       0         mndnba            
         Hy

In [8]:
# [8] 개념 단위 정규화·중복 제거
VISIT_PAT = re.compile(
    r"\s*\((sleep heart health study )?visit (one|two)\s*\(shhs[12]\)\)|\s*\(shhs[12]\)", re.I)

def norm_label(s):
    s = VISIT_PAT.sub("", s if isinstance(s, str) else "")
    return re.sub(r"\s+", " ", s).strip()

inc = []
for name in inv:
    sub = inv[name][inv[name]["분류"].str.startswith("개별")].copy()
    sub["개념라벨"] = sub["display_name"].map(norm_label)
    sub["cohort"] = name
    inc.append(sub)
inc = pd.concat(inc, ignore_index=True)

concepts = (inc.groupby(inc["개념라벨"].str.lower())
            .agg(개념라벨=("개념라벨", "first"),
                 n변수=("id", "count"),
                 변수목록=("id", lambda s: ";".join(sorted(set(s))[:8])),
                 코호트=("cohort", lambda s: "+".join(sorted(set(s)))),
                 폴더예시=("folder", "first"))
            .reset_index(drop=True))

print(f"포함 변수 {len(inc)}개 → 고유 개념 {len(concepts)}개")
print(concepts["코호트"].value_counts().to_string())
print(concepts.sample(10, random_state=1)[["개념라벨", "n변수", "코호트"]].to_string(index=False))

포함 변수 685개 → 고유 개념 584개
코호트
SHHS         440
MESA         143
MESA+SHHS      1
                                                                                                                        개념라벨  n변수  코호트
                                                                                   Age at Sleep Heart Health Study Visit One    2 SHHS
                                                                                     Supine arm systolic blood pressure (BP)    1 SHHS
                                                      Percentage of total sleep duration in REM from type II polysomnography    1 MESA
  Apnea-Hypopnea Index (AHI) >= 3% - number of [all apneas] and [hypopneas with >= 3% oxygen desaturation] per hour of sleep    1 SHHS
                                  REM Sleep Latency: the interval between the first sleep epoch and REM sleep including wake    1 MESA
                                           Number of Percutaneous transluminal coronary angioplasties (PTCAs) S

In [9]:
# [9] EBI OLS4 후보 개념 검색 (HPO / MONDO / SNOMED / EFO) — API 키 불필요
import requests, json, time

OLS_URL = "https://www.ebi.ac.uk/ols4/api/search"
ONTOS = ["hp", "mondo", "snomed", "efo"]
CACHE = OUT_DIR / "ols_cache.json"
cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}

def ols_search(q, rows=5):
    key = q.lower()
    if key in cache:
        return cache[key]
    r = requests.get(OLS_URL, params={"q": q, "ontology": ONTOS, "rows": rows,
                                      "fieldList": "iri,label,obo_id,ontology_name,short_form"},
                     timeout=30)
    r.raise_for_status()
    docs = r.json()["response"]["docs"]
    out = [{"obo_id": d.get("obo_id") or d.get("short_form", ""),
            "label": d.get("label", ""), "ontology": d.get("ontology_name", "")}
           for d in docs]
    cache[key] = out
    time.sleep(0.15)
    return out

results, fails = [], []
for i, row in concepts.iterrows():
    try:
        hits = ols_search(row["개념라벨"])
    except Exception as e:
        hits = []; fails.append((row["개념라벨"], str(e)))
    if hits:
        for rank, h in enumerate(hits, 1):
            results.append({**row.to_dict(), "순위": rank, **h})
    else:
        results.append({**row.to_dict(), "순위": 0, "obo_id": "", "label": "(후보없음)", "ontology": ""})
    if (i + 1) % 50 == 0:
        CACHE.write_text(json.dumps(cache))
        print(f"진행: {i+1}/{len(concepts)} (실패 {len(fails)})")
CACHE.write_text(json.dumps(cache))

res = pd.DataFrame(results)
out_map = OUT_DIR / "OLS_mapping_candidates.xlsx"
res.to_excel(out_map, index=False)
n_hit = res.query("순위 == 1").shape[0]
print(f"\n후보 1개 이상 확보: {n_hit}/{len(concepts)}개 개념")
if fails:
    print("네트워크 실패:", len(fails), "건 — 재실행하면 캐시 이후부터 재시도됩니다")
print("저장:", out_map.resolve())

진행: 50/584 (실패 0)
진행: 100/584 (실패 0)
진행: 150/584 (실패 0)
진행: 200/584 (실패 0)
진행: 250/584 (실패 0)
진행: 300/584 (실패 0)
진행: 350/584 (실패 0)
진행: 400/584 (실패 0)
진행: 450/584 (실패 0)
진행: 500/584 (실패 0)
진행: 550/584 (실패 0)

후보 1개 이상 확보: 67/584개 개념
저장: /mnt/e/Claude Cowork/SHHS Ontology/ontology_outputs/OLS_mapping_candidates.xlsx


In [10]:
# [10] 질의 정제 체인으로 재검색
MED_CANON = [
    (re.compile(r"anti-?arrhythmics?, class \w+", re.I), "antiarrhythmic agent"),
    (re.compile(r"angiotensin type 2 antagonists?", re.I), "angiotensin II receptor antagonist"),
    (re.compile(r"ankle-arm blood pressure.*index", re.I), "ankle brachial pressure index"),
    (re.compile(r"calcium-?channel blocker", re.I), "calcium channel blocker"),
]
STRIP_PHRASES = re.compile(
    r"\(.*?\)|\[.*?\]|since baseline|at sleep heart health study visit (one|two)|"
    r"from type ii polysomnography|w/ arousals?|(with|without|plus) diuretics?|\?", re.I)
PREFIXES = re.compile(
    r"^(any|number of|days to first|days to|time to|has|usual|frequency of|"
    r"history of|combinations of)\s+", re.I)

def make_queries(label):
    qs = []
    t = label if isinstance(label, str) else ""
    for pat, rep in MED_CANON:
        if pat.search(t):
            qs.append(rep)
    head = re.split(r"\s*[:=]\s*|\s+-\s+", t)[0]   # 구분자 앞 머리 어구
    for cand in (t, head):
        c = STRIP_PHRASES.sub(" ", cand)
        c = PREFIXES.sub("", re.sub(r"\s+", " ", c).strip()).strip(" ,.-")
        if c and c.lower() not in [q.lower() for q in qs]:
            qs.append(c)
    last = qs[-1] if qs else ""
    if len(last.split()) > 6:
        qs.append(" ".join(last.split()[:4]))
    return qs

def search_chain(label):
    for step, q in enumerate(make_queries(label), 1):
        try:
            hits = ols_search(q)
        except Exception:
            hits = []
        if hits:
            return q, step, hits
    return "", 0, []

results = []
for i, row in concepts.iterrows():
    q, step, hits = search_chain(row["개념라벨"])
    base = {**row.to_dict(), "사용질의": q, "질의단계": step}
    if hits:
        for rank, h in enumerate(hits, 1):
            results.append({**base, "순위": rank, **h})
    else:
        results.append({**base, "순위": 0, "obo_id": "", "label": "(후보없음)", "ontology": ""})
    if (i + 1) % 50 == 0:
        CACHE.write_text(json.dumps(cache))
        print(f"진행: {i+1}/{len(concepts)}")
CACHE.write_text(json.dumps(cache))

res = pd.DataFrame(results)
res["정확일치"] = res["label"].str.lower() == res["사용질의"].str.lower()
res["선택"] = ""      # 검수: 채택할 후보에 O
res["최종개념ID"] = ""  # 후보에 없으면 직접 기입
res["코멘트"] = ""

out_map = OUT_DIR / "OLS_mapping_candidates_v2.xlsx"
res.to_excel(out_map, index=False)

ok = res.query("순위 == 1")
print(f"\n후보 확보: {len(ok)}/{len(concepts)}개 개념 "
      f"(정확일치 {int(res['정확일치'].sum())}건 포함)")
miss = res.query("순위 == 0")
print(f"여전히 실패: {len(miss)}개 — 상위 20개:")
print(miss["개념라벨"].head(20).to_string(index=False))

진행: 50/584
진행: 100/584
진행: 150/584
진행: 200/584
진행: 250/584
진행: 300/584
진행: 350/584
진행: 400/584
진행: 450/584
진행: 500/584
진행: 550/584

후보 확보: 334/584개 개념 (정확일치 152건 포함)
여전히 실패: 250개 — 상위 20개:
                                     Arousal Index
Arousal Index (Non-rapid eye movement sleep (NR...
Arousal Index (NREM): Number of arousals per ho...
Arousal Index (NREM, Non-Supine): Number of aro...
Arousal Index (NREM, Supine): Number of arousal...
    Arousal Index (Rapid eye movement sleep (REM))
Arousal Index (REM): Number of arousals per hou...
Arousal Index (REM, Non-Supine): Number of arou...
Arousal Index (REM, Supine): Number of arousals...
Arousal Index: Number of arousals per hour of s...
Arousal Index: Number of arousals per hour of s...
Arousals per hour (Non-rapid eye movement sleep...
Arousals per hour (Non-rapid eye movement sleep...
Arousals per hour (Rapid eye movement sleep (RE...
Arousals per hour (Rapid eye movement sleep (RE...
          Aspirin From 280804 (Anti-Inflam Age

In [11]:
# [11] LLM 보조 큐레이션 적용 → 매핑 마스터
cur = pd.read_csv(OUT_DIR / "manual_curation.csv", encoding="utf-8-sig")
try:
    res
except NameError:
    res = pd.read_excel(OUT_DIR / "OLS_mapping_candidates_v2.xlsx")

def search_first(chain):
    for q in [s.strip() for s in str(chain).split("|") if s.strip() and s != "nan"]:
        try:
            hits = ols_search(q)
        except Exception:
            hits = []
        if hits:
            return q, hits
    return "", []

auto = res[res["순위"] >= 1].copy()
auto["판정"] = "자동매핑후보"

rows = []
for i, r in cur.iterrows():
    q, hits = search_first(r["큐레이션질의"])
    base = {"개념라벨": r["개념라벨"], "폴더예시": r["폴더예시"], "코호트": r["코호트"],
            "판정": r["판정"], "규칙": r["규칙"], "사용질의": q}
    if hits:
        for rank, h in enumerate(hits, 1):
            rows.append({**base, "순위": rank, **h})
    else:
        rows.append({**base, "순위": 0, "obo_id": "", "label": "(후보없음)", "ontology": ""})
    if (i + 1) % 50 == 0:
        CACHE.write_text(json.dumps(cache))
        print(f"진행 {i+1}/{len(cur)}")
CACHE.write_text(json.dumps(cache))
man = pd.DataFrame(rows)

master = pd.concat([auto, man], ignore_index=True)
for col in ("선택", "최종개념ID", "코멘트"):
    if col not in master.columns:
        master[col] = ""

out_master = OUT_DIR / "OSA_mapping_master.xlsx"
per_concept = (master.groupby("개념라벨")
               .agg(판정=("판정", "first"), 후보=("순위", "max")).reset_index())
summ = (per_concept.groupby("판정")
        .agg(개념수=("개념라벨", "count"),
             후보확보=("후보", lambda s: int((s > 0).sum()))).reset_index())
with pd.ExcelWriter(out_master, engine="openpyxl") as xw:
    master.to_excel(xw, sheet_name="매핑마스터", index=False)
    summ.to_excel(xw, sheet_name="요약", index=False)

print(summ.to_string(index=False))
print("저장:", out_master.resolve())

진행 50/250
진행 100/250
진행 150/250
진행 200/250
진행 250/250
    판정  개념수  후보확보
  관리변수   18     0
  설문항목   62    62
 신규클래스  104   104
자동매핑후보  334   334
   재질의   41    40
  파생변수   25    25
저장: /mnt/e/Claude Cowork/SHHS Ontology/ontology_outputs/OSA_mapping_master.xlsx
